# 🔬 Applying R and AI for Automated Reference Interval Review
**ASCP 2026 Interactive Guided Walkthrough — Multi-Site Serum Calcium Harmonization Case Study**

Welcome! Every participant connected to this notebook runs in a **private, isolated cloud workspace**—your prompts, code, and outputs will never interfere with other simultaneous attendees.

### How to Use This Interactive Workspace
1. **Run Setup Now:** Click the **Play (▶)** button on **Step 0** immediately below to install the R `refineR` package (~15 seconds) and load our 6-month multi-site LIS Calcium extract (`26,001` adult results across `4` sites and `2` analyzer platforms).
2. **Generate Code with Gemini:** At each of the 4 Walkthrough Steps, click **`📋 Copy Prompt`**, then click **`generate`** inside the empty code cell right beneath it, paste the prompt, and run the cell.
3. **One-Click Sync Guardrails:** If your AI-generated code hits a snag or names columns differently, click **Play (▶)** on the **`🛡️ [Guardrail Checkpoint]`** cell beneath that step to instantly synchronize canonical tables for the next step.

In [ ]:
#@title 🚀 Step 0: Environment Setup, `refineR` Installer & Calcium Dataset Loader (Click Play First!) { display-mode: "form" }
import os
import re
import warnings
from io import BytesIO
import base64
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 15)
pd.set_option("display.width", 1000)

print("⏳ [1/3] Loading `rpy2` bridge and installing pre-compiled R `refineR` binary (~15 seconds)...")
_HAS_RPY2 = False
_HAS_REFINER = False
try:
    from rpy2 import robjects
    from rpy2.robjects.packages import importr
    try:
        get_ipython().run_line_magic("load_ext", "rpy2.ipython")
    except Exception:
        pass
    _HAS_RPY2 = True
    robjects.r('''
    codename <- tryCatch({
      out <- system("grep ^VERSION_CODENAME= /etc/os-release | cut -d= -f2", intern = TRUE)
      gsub('"', "", trimws(out[1]))
    }, error = function(e) "jammy")
    if (is.na(codename) || nchar(codename) == 0) codename <- "jammy"
    p3m_url <- paste0("https://packagemanager.posit.co/cran/__linux__/", codename, "/latest")
    options(repos = c(P3M = p3m_url, CRAN = "https://cloud.r-project.org"))
    if (!requireNamespace("refineR", quietly = TRUE)) {
      install.packages("refineR", quiet = TRUE)
    }
    library(refineR)
    ''')
    importr("refineR")
    _HAS_REFINER = True
    print("✅ R `refineR` package loaded via `rpy2`!")
except Exception as e:
    print(f"⚠️ R/refineR unavailable; nonparametric 2.5th–97.5th percentile fallback active: {e}")

# -------------------------------------------------------------------------
# Core Clinical & Statistical Helper Functions
# -------------------------------------------------------------------------
_code_pat = re.compile(r"^[A-Z0-9][A-Z0-9.]*$", re.I)

ICD_DESCRIPTORS = {
    "Z00.00":  "General adult medical exam (Normal)",
    "Z12.31":  "Screening mammogram (Normal)",
    "Z12.11":  "Colon neoplasm screening (Normal)",
    "Z12.5":   "Prostate neoplasm screening (Normal)",
    "Z01.818": "Preprocedural lab exam (Normal)",
    "Z23":     "Encounter for immunization (Normal)",
    "I10":     "Essential hypertension (Unbiased)",
    "E78.5":   "Hyperlipidemia, unspecified (Unbiased)",
    "E11.9":   "Type 2 diabetes, uncomplicated (Unbiased)",
    "R07.9":   "Chest pain, unspecified (Unbiased)",
    "M17.11":  "Primary knee osteoarthritis (Unbiased)",
    "E55.9":   "Vitamin D deficiency (Hypocalcemia)",
    "N18.4":   "Chronic kidney disease, stage 4 (Hypocalcemia)",
    "N18.6":   "End stage renal disease (Hypocalcemia)",
    "E88.09":  "Hypoalbuminemia / protein disorder (Artifactual Low Ca)",
    "A41.9":   "Sepsis, unspecified organism (Critical Illness Low Ca)",
    "K85.90":  "Acute pancreatitis (Saponification Low Ca)",
    "E21.0":   "Primary hyperparathyroidism (Hypercalcemia)",
    "F31.9":   "Bipolar disorder — Lithium-associated PTH shift (Unexpected)",
    "D86.9":   "Sarcoidosis — 1,25-(OH)2D mediated (Hypercalcemia)",
    "C90.00":  "Multiple myeloma — Osteolytic (Hypercalcemia)",
    "C79.51":  "Secondary malignant neoplasm of bone (Hypercalcemia)",
}

def split_codes(s: str):
    """Parse semicolon/comma/whitespace-delimited ICD-10 codes."""
    if pd.isna(s):
        return []
    toks = re.split(r"[\s,;]+", str(s))
    return [
        t.strip().upper()
        for t in toks
        if t and t.strip().upper() != "NS" and _code_pat.fullmatch(t.strip().upper())
    ]

def icd_root(code: str) -> str:
    """Extract 3-character ICD-10 root category."""
    return re.sub(r"\.", "", str(code))[:3].upper()

def bh_fdr(p: np.ndarray) -> np.ndarray:
    """Benjamini-Hochberg adjusted p-values (q-values)."""
    q = np.full_like(p, np.nan, dtype=float)
    mask = np.isfinite(p)
    m = mask.sum()
    if m == 0:
        return q
    idx = np.where(mask)[0]
    p_valid = p[idx]
    order = np.argsort(p_valid)
    ranked = p_valid[order]
    bh = ranked * m / np.arange(1, m + 1)
    bh_rev = np.minimum.accumulate(bh[::-1])[::-1]
    adj = np.empty_like(p_valid)
    adj[order] = np.clip(bh_rev, 0.0, 1.0)
    q[idx] = adj
    return q

def cluster_ols(y: pd.Series, X: pd.DataFrame, clusters: pd.Series):
    """OLS with MRN-clustered standard errors using statsmodels."""
    y = pd.to_numeric(y, errors="coerce").astype("float64")
    Xc = sm.add_constant(X, has_constant="add").apply(pd.to_numeric, errors="coerce").astype("float64")
    row_mask = y.notna() & Xc.notna().all(axis=1)
    y_fit, X_fit, clus_fit = y.loc[row_mask], Xc.loc[row_mask], clusters.loc[row_mask]
    res = sm.OLS(y_fit, X_fit).fit(cov_type="cluster", cov_kwds={"groups": clus_fit})
    ci = res.conf_int(alpha=0.05)
    out = pd.DataFrame({
        "term": res.params.index,
        "coef": res.params.values,
        "std_err": res.bse.values,
        "z_or_t": res.tvalues.values,
        "p_value": res.pvalues.values,
        "ci_lower_95": ci.iloc[:, 0].values,
        "ci_upper_95": ci.iloc[:, 1].values,
    })
    return res, out

def run_refiner_ca(df_sub: pd.DataFrame) -> dict:
    """Runs R refineR on Result_Value (or nonparametric fallback if R is unavailable)."""
    dedup = globals().get("DEDUPLICATE_FIRST_MRN_FOR_REFINER", False)
    work = df_sub.sort_values("Date_of_Sample").drop_duplicates("MRN") if dedup else df_sub
    values = work["Result_Value"].dropna().values.astype(float)
    if values.size == 0:
        return {"low": np.nan, "high": np.nan, "method": "No data", "n_used": 0}
    if _HAS_RPY2 and _HAS_REFINER:
        try:
            robjects.globalenv["x_ca"] = robjects.FloatVector(values)
            robjects.r('''
            fit_ca <- refineR::findRI(Data = x_ca)
            ri_ca  <- refineR::getRI(fit_ca, Percentile = c(0.025, 0.975))
            low_ca <- as.numeric(ri_ca$PointEst[which.min(abs(ri_ca$Percentile - 0.025))])
            high_ca <- as.numeric(ri_ca$PointEst[which.min(abs(ri_ca$Percentile - 0.975))])
            ''')
            lo = round(float(robjects.r("low_ca")[0]), 1)
            hi = round(float(robjects.r("high_ca")[0]), 1)
            return {"low": lo, "high": hi, "method": "refineR", "n_used": len(values)}
        except Exception:
            pass
    lo, hi = np.percentile(values, [2.5, 97.5])
    return {"low": round(float(lo), 1), "high": round(float(hi), 1), "method": "Nonparametric (Fallback)", "n_used": len(values)}

# -------------------------------------------------------------------------
# Self-Healing Canonical Pipeline Helpers (Prevents Column/Schema KeyErrors)
# -------------------------------------------------------------------------
def _has_cols(var_name: str, req_cols: list) -> bool:
    obj = globals().get(var_name, None)
    if not isinstance(obj, pd.DataFrame) or obj.empty:
        return False
    return all(c in obj.columns for c in req_cols)

def _sync_step1(force: bool = False):
    g = globals()
    req_adult = ["MRN", "Specimen_ID", "Patient_Age", "Date_of_Sample", "Performance_Site", "Analyzer_Type", "ICD10_Codes", "Result_Value", "Manufacturer_Stated_RI", "icd_list"]
    req_sum = ["Performance_Site", "Analyzer_Type", "Manufacturer_Stated_RI", "n_results", "n_mrn", "Tests_per_MRN", "serial_inpatients", "mean_draws_serial", "raw_mean", "raw_sd", "raw_p2_5", "raw_p97_5"]
    if not force and _has_cols("df_adult", req_adult) and _has_cols("site_summary_df", req_sum):
        return
    df_a = pd.read_csv("/content/ascp_electrolyte_data.csv")
    for col in ["MRN", "Specimen_ID", "Performance_Site", "Analyzer_Type", "ICD10_Codes", "Analyte_Name", "Result_Units", "Manufacturer_Stated_RI"]:
        df_a[col] = df_a[col].fillna("").astype(str).str.strip()
    df_a = df_a[(df_a["Patient_Age"] >= 18) & (df_a["Analyte_Name"] == "Calcium")].copy().reset_index(drop=True)
    df_a["icd_list"] = df_a["ICD10_Codes"].apply(split_codes)

    mrn_day_audit = (
        df_a.groupby(["Performance_Site", "MRN"], as_index=False)
        .agg(n_draws=("Result_Value", "count"), n_distinct_days=("Date_of_Sample", "nunique"))
    )
    serial_summary = (
        mrn_day_audit[mrn_day_audit["n_draws"] > 1]
        .groupby("Performance_Site", as_index=False)
        .agg(serial_inpatients=("MRN", "count"), mean_draws_serial=("n_draws", "mean"), max_draws=("n_draws", "max"))
    )
    s_sum = (
        df_a.groupby(["Performance_Site", "Analyzer_Type", "Manufacturer_Stated_RI"], as_index=False)
        .agg(
            n_results=("Result_Value", "count"),
            n_mrn=("MRN", "nunique"),
            raw_mean=("Result_Value", "mean"),
            raw_sd=("Result_Value", "std"),
            raw_p2_5=("Result_Value", lambda x: np.percentile(x, 2.5)),
            raw_p97_5=("Result_Value", lambda x: np.percentile(x, 97.5)),
        )
        .merge(serial_summary, on="Performance_Site", how="left")
    )
    s_sum["serial_inpatients"] = s_sum["serial_inpatients"].fillna(0).astype(int)
    s_sum["mean_draws_serial"] = s_sum["mean_draws_serial"].fillna(1.0).round(2)
    s_sum["Tests_per_MRN"] = (s_sum["n_results"] / s_sum["n_mrn"]).round(2)
    for c in ["raw_mean", "raw_sd", "raw_p2_5", "raw_p97_5"]:
        s_sum[c] = s_sum[c].round(2)
    g["df_adult"] = df_a
    g["site_summary_df"] = s_sum

def _sync_step2(force: bool = False):
    _sync_step1(force=False)
    g = globals()
    req_site = ["Performance_Site", "Analyzer_Type", "% Effect Size", "Shift_mg_dL", "CI_95_Low", "CI_95_High"]
    req_codes = ["ICD10_Code", "Clinical_Descriptor", "n_mrn", "Effect_mg_dL", "CI_Low", "CI_High", "q_value", "Flagged_Confounder"]
    if (
        not force
        and _has_cols("df_clean", ["MRN", "Performance_Site", "Analyzer_Type", "Result_Value", "Manufacturer_Stated_RI", "icd_list"])
        and _has_cols("site_effects_pre_df", req_site)
        and _has_cols("site_effects_post_df", req_site)
        and _has_cols("sig_codes_df", req_codes)
        and _has_cols("unflagged_codes_df", req_codes)
        and _has_cols("factorial_2x2_df", ["term", "coef", "%_of_RI_Width"])
        and _has_cols("logit_site_df", ["Term", "Odds_Ratio_Abnormal_Flag"])
        and _has_cols("cohort_sizes_df", ["Metric", "Value"])
    ):
        return

    df_a = g["df_adult"]
    sites_sorted = sorted(df_a["Performance_Site"].unique())
    non_ref_sites = [s for s in sites_sorted if s != LAB_REF]

    X_site_pre = pd.DataFrame({f"SITE__{s}": (df_a["Performance_Site"] == s).astype(float) for s in non_ref_sites}, index=df_a.index)
    res_pre, tab_pre = cluster_ols(df_a["Result_Value"], X_site_pre, df_a["MRN"])

    def _fmt_site_table(df_source: pd.DataFrame, ols_tab: pd.DataFrame, label: str) -> pd.DataFrame:
        rows = []
        for s in sites_sorted:
            sub = df_source[df_source["Performance_Site"] == s]
            an_type = sub["Analyzer_Type"].iloc[0]
            if s == LAB_REF:
                coef, se, zt, pval, cil, ciu = 0.0, 0.0, 0.0, 0.0, 0.0, 0.0
            else:
                r = ols_tab[ols_tab["term"] == f"SITE__{s}"].iloc[0]
                coef, se, zt, pval, cil, ciu = r["coef"], r["std_err"], r["z_or_t"], r["p_value"], r["ci_lower_95"], r["ci_upper_95"]
            rows.append({
                "Cohort": label,
                "Performance_Site": s,
                "Analyzer_Type": an_type,
                "n_results": len(sub),
                "n_mrn": sub["MRN"].nunique(),
                "% Effect Size": round((coef / REF_RI_WIDTH) * 100.0, 1),
                "% lower CI": round((cil / REF_RI_WIDTH) * 100.0, 1),
                "% upper CI": round((ciu / REF_RI_WIDTH) * 100.0, 1),
                "Shift_mg_dL": round(coef, 3),
                "Clustered_SE": round(se, 3),
                "p_value": round(pval, 4),
                "CI_95_Low": round(cil, 3),
                "CI_95_High": round(ciu, 3),
            })
        return pd.DataFrame(rows)

    site_pre = _fmt_site_table(df_a, tab_pre, "Pre-Exclusion (All ICD-10)")

    X_fwl = sm.add_constant(X_site_pre, has_constant="add").values
    y_resid = res_pre.resid.values
    Q, _ = np.linalg.qr(X_fwl, mode="reduced")

    _, clus_idx = np.unique(df_a["MRN"].values, return_inverse=True)
    n_clus = clus_idx.max() + 1
    n_obs, k_params = X_fwl.shape
    dof_scale = (n_clus / (n_clus - 1.0)) * ((n_obs - 1.0) / max(n_obs - k_params - 1.0, 1.0))

    code_mrn_map, code_row_map = {}, {}
    for idx, (m_val, c_list) in enumerate(zip(df_a["MRN"], df_a["icd_list"])):
        for c in c_list:
            code_mrn_map.setdefault(c, set()).add(m_val)
            code_row_map.setdefault(c, []).append(idx)

    eligible_codes = sorted([c for c, mset in code_mrn_map.items() if len(mset) >= CODE_MIN_MRN])
    code_rows = []
    for c in eligible_codes:
        d = np.zeros(n_obs, dtype=float)
        d[code_row_map[c]] = 1.0
        d_tilde = d - Q @ (Q.T @ d)
        denom = np.dot(d_tilde, d_tilde)
        if denom < 1e-10:
            continue
        beta_c = np.dot(d_tilde, y_resid) / denom
        e_c = y_resid - d_tilde * beta_c
        clus_sums = np.bincount(clus_idx, weights=d_tilde * e_c, minlength=n_clus)
        se_c = np.sqrt(np.sum(clus_sums ** 2) * dof_scale) / denom
        t_stat = beta_c / se_c if se_c > 0 else 0.0
        p_val = 2.0 * (1.0 - stats.t.cdf(abs(t_stat), df=n_clus - 1))
        code_rows.append({
            "ICD10_Code": c,
            "Clinical_Descriptor": ICD_DESCRIPTORS.get(c, "Clinical diagnosis"),
            "n_results": len(code_row_map[c]),
            "n_mrn": len(code_mrn_map[c]),
            "Effect_mg_dL": round(beta_c, 3),
            "%_RI_Width": round((beta_c / REF_RI_WIDTH) * 100.0, 1),
            "Clustered_SE": round(se_c, 3),
            "CI_Low": round(beta_c - 1.96 * se_c, 3),
            "CI_High": round(beta_c + 1.96 * se_c, 3),
            "p_value": p_val,
        })

    c_df = pd.DataFrame(code_rows)
    c_df["q_value"] = bh_fdr(c_df["p_value"].values)
    c_df["Flagged_Confounder"] = (
        (c_df["q_value"] < FDR_THRESHOLD) &
        (c_df["Effect_mg_dL"].abs() >= EFFECT_MIN_UNITS)
    )
    c_df = c_df.sort_values("Effect_mg_dL").reset_index(drop=True)

    flagged_codes_set = set(c_df.loc[c_df["Flagged_Confounder"], "ICD10_Code"])
    df_a["n_flagged"] = df_a["icd_list"].apply(lambda lst: sum(1 for c in lst if c in flagged_codes_set))

    keep_mask = (df_a["n_flagged"] == 0) if EXCLUSION_POLICY == "strict" else (df_a["n_flagged"] <= 1)
    df_c = df_a[keep_mask].copy().reset_index(drop=True)
    df_e = df_a[~keep_mask].copy().reset_index(drop=True)

    X_site_post = pd.DataFrame({f"SITE__{s}": (df_c["Performance_Site"] == s).astype(float) for s in non_ref_sites}, index=df_c.index)
    _, tab_post = cluster_ols(df_c["Result_Value"], X_site_post, df_c["MRN"])
    site_post = _fmt_site_table(df_c, tab_post, "Post-Exclusion (Bias-Controlled)")

    X_2x2 = pd.DataFrame({
        "Analyzer_Omega_vs_Alpha": (df_c["Analyzer_Type"] == "Analyzer Omega").astype(float),
        "Inpatient_Hospital_vs_Outpatient": df_c["Performance_Site"].str.contains("Inpatient").astype(float),
    }, index=df_c.index)
    _, fact_df = cluster_ols(df_c["Result_Value"], X_2x2, df_c["MRN"])
    fact_df["%_of_RI_Width"] = ((fact_df["coef"] / REF_RI_WIDTH) * 100.0).round(1)

    def _is_out_of_mfg(row):
        lo, hi = [float(x) for x in row["Manufacturer_Stated_RI"].split("-")]
        return int(row["Result_Value"] < lo or row["Result_Value"] > hi)

    df_a["Abnormal_Mfg_Flag"] = df_a.apply(_is_out_of_mfg, axis=1)
    X_logit = sm.add_constant(X_site_pre, has_constant="add")
    logit_fit = sm.Logit(df_a["Abnormal_Mfg_Flag"], X_logit).fit(disp=False)
    logit_ci = np.exp(logit_fit.conf_int())
    logit_df = pd.DataFrame({
        "Term": [t.replace("SITE__", "") for t in logit_fit.params.index],
        "Odds_Ratio_Abnormal_Flag": np.exp(logit_fit.params.values).round(2),
        "OR_95_Low": logit_ci.iloc[:, 0].values.round(2),
        "OR_95_High": logit_ci.iloc[:, 1].values.round(2),
        "p_value": logit_fit.pvalues.values.round(4),
    })

    c_sizes = pd.DataFrame([
        {"Metric": "Total Adult Calcium Results", "Value": f"{len(df_a):,}"},
        {"Metric": "Total Unique Adult MRNs", "Value": f"{df_a['MRN'].nunique():,}"},
        {"Metric": "Clean Reference Results Retained", "Value": f"{len(df_c):,} ({100*len(df_c)/len(df_a):.1f}%)"},
        {"Metric": "Clean Unique MRNs Retained", "Value": f"{df_c['MRN'].nunique():,}"},
        {"Metric": "Flagged Pathological / Drug Codes Excluded", "Value": f"{len(flagged_codes_set)} of {len(c_df)} tested codes"},
    ])

    g["site_effects_pre_df"] = site_pre
    g["site_effects_post_df"] = site_post
    g["code_effects_df"] = c_df
    g["sig_codes_df"] = c_df[c_df["Flagged_Confounder"]].copy().reset_index(drop=True)
    g["unflagged_codes_df"] = c_df[~c_df["Flagged_Confounder"]].copy().reset_index(drop=True)
    g["df_clean"] = df_c
    g["df_excluded"] = df_e
    g["factorial_2x2_df"] = fact_df
    g["logit_site_df"] = logit_df
    g["cohort_sizes_df"] = c_sizes

def _sync_step3(force: bool = False):
    _sync_step2(force=False)
    g = globals()
    req_ri = ["Performance_Site", "Analyzer_Type", "n_clean", "Mfg_Stated_RI", "Raw_Unfiltered_95%", "refineR_Derived_RI", "refineR_Low", "refineR_High", "Mfg_Low", "Mfg_High", "Healthy_Inside_Mfg_%", "CLSI_N20_Pass_Rate_%", "Method"]
    req_clin = ["Analyzer_Type", "E21_0_Cases", "Mfg_Upper_Limit", "refineR_Upper_Limit", "Sensitivity_Using_Mfg_RI_%", "Sensitivity_Using_refineR_RI_%", "Missed_Mild_Hypercalcemia_Cases"]
    if not force and _has_cols("ri_per_site_df", req_ri) and _has_cols("clinical_impact_df", req_clin):
        return

    df_c = g["df_clean"]
    df_a = g["df_adult"]
    sites_sorted = sorted(df_c["Performance_Site"].unique())
    mc_rng = np.random.default_rng(2026)
    site_ri_rows = []
    for s in sites_sorted:
        sub_c = df_c[df_c["Performance_Site"] == s]
        sub_raw = df_a[df_a["Performance_Site"] == s]
        an_type = sub_c["Analyzer_Type"].iloc[0]
        mfg_str = sub_c["Manufacturer_Stated_RI"].iloc[0]
        mfg_lo, mfg_hi = [float(x) for x in str(mfg_str).split("-")]

        ri_clean = run_refiner_ca(sub_c)
        raw_lo, raw_hi = np.round(np.percentile(sub_raw["Result_Value"], [2.5, 97.5]), 1)

        healthy_vals = sub_c["Result_Value"].values
        pct_inside_mfg = 100.0 * np.mean((healthy_vals >= mfg_lo) & (healthy_vals <= mfg_hi))
        sim_draws = mc_rng.choice(healthy_vals, size=(1000, 20), replace=True)
        clsi_pass_rate = 100.0 * np.mean(np.sum((sim_draws >= mfg_lo) & (sim_draws <= mfg_hi), axis=1) >= 18)

        site_ri_rows.append({
            "Performance_Site": s,
            "Analyzer_Type": an_type,
            "n_clean": ri_clean["n_used"],
            "Mfg_Stated_RI": f"{mfg_lo:.1f} – {mfg_hi:.1f}",
            "Raw_Unfiltered_95%": f"{raw_lo:.1f} – {raw_hi:.1f}",
            "refineR_Derived_RI": f"{ri_clean['low']:.1f} – {ri_clean['high']:.1f}",
            "refineR_Low": ri_clean["low"],
            "refineR_High": ri_clean["high"],
            "Mfg_Low": mfg_lo,
            "Mfg_High": mfg_hi,
            "Healthy_Inside_Mfg_%": round(pct_inside_mfg, 1),
            "CLSI_N20_Pass_Rate_%": round(clsi_pass_rate, 1),
            "Method": ri_clean["method"],
        })

    ri_df = pd.DataFrame(site_ri_rows)
    alpha_outpt = run_refiner_ca(df_c[df_c["Performance_Site"] == "Regional Outpatient Clinic"])
    omega_outpt = run_refiner_ca(df_c[df_c["Performance_Site"] == "Downtown Outpatient Clinic"])

    phpt_df = df_a[df_a["icd_list"].apply(lambda lst: "E21.0" in lst)].copy()
    clin_rows = []
    for an_type, mfg_hi, ref_hi in [
        ("Analyzer Alpha", 10.4, alpha_outpt["high"]),
        ("Analyzer Omega", 10.5, omega_outpt["high"]),
    ]:
        sub_p = phpt_df[phpt_df["Analyzer_Type"] == an_type]
        sens_mfg = 100.0 * np.mean(sub_p["Result_Value"] > mfg_hi)
        sens_ref = 100.0 * np.mean(sub_p["Result_Value"] > ref_hi)
        missed_zone = np.sum((sub_p["Result_Value"] > ref_hi) & (sub_p["Result_Value"] <= mfg_hi))
        clin_rows.append({
            "Analyzer_Type": an_type,
            "E21_0_Cases": len(sub_p),
            "Mfg_Upper_Limit": f"{mfg_hi:.1f} mg/dL",
            "refineR_Upper_Limit": f"{ref_hi:.1f} mg/dL",
            "Sensitivity_Using_Mfg_RI_%": round(sens_mfg, 1),
            "Sensitivity_Using_refineR_RI_%": round(sens_ref, 1),
            "Missed_Mild_Hypercalcemia_Cases": int(missed_zone),
        })
    g["ri_per_site_df"] = ri_df
    g["clinical_impact_df"] = pd.DataFrame(clin_rows)

def _fig_to_b64(fig_obj) -> str:
    buf = BytesIO()
    fig_obj.savefig(buf, format="png", dpi=145, bbox_inches="tight")
    return base64.b64encode(buf.getvalue()).decode("utf-8")

def launch_html_dashboard(fig1_or_html=None, fig2=None, filename: str = "/content/ASCP_Calcium_QA_Dashboard.html"):
    """
    Polymorphic QA Dashboard Builder & New-Tab Launcher:
      - Call `launch_html_dashboard(fig1, fig2)` with two matplotlib Figures (eliminates raw HTML in Gemini cells!)
      - Or call `launch_html_dashboard(html_string)` with a pre-built HTML string.
    """
    from IPython.display import HTML, display
    os.makedirs(os.path.dirname(filename), exist_ok=True)

    if isinstance(fig1_or_html, str) and "<html" in fig1_or_html.lower():
        html_content = fig1_or_html
    else:
        # Ensure all summary tables exist (without overwriting user's Gemini tables if they want to inspect them,
        # or backfilling canonical tables for dashboard tabs)
        if isinstance(fig1_or_html, (list, tuple)) and len(fig1_or_html) >= 2 and fig2 is None:
            f1, f2 = fig1_or_html[0], fig1_or_html[1]
        else:
            f1, f2 = fig1_or_html, fig2

        # Ensure canonical tables are ready for KPIs and Tabs 2-4
        _sync_step3(force=False)
        g = globals()

        b64_1 = _fig_to_b64(f1) if hasattr(f1, "savefig") else ""
        b64_2 = _fig_to_b64(f2) if hasattr(f2, "savefig") else b64_1
        if not b64_1 and b64_2:
            b64_1 = b64_2

        def _tbl_html(df_in: pd.DataFrame) -> str:
            return df_in.to_html(index=False, classes="data-table", border=0, na_rep="—")

        fact_df = g["factorial_2x2_df"]
        omega_bias_val = fact_df.loc[fact_df["term"] == "Analyzer_Omega_vs_Alpha", "coef"].iloc[0]
        inpt_shift_val = fact_df.loc[fact_df["term"] == "Inpatient_Hospital_vs_Outpatient", "coef"].iloc[0]
        s_sum = g["site_summary_df"]
        site_sum_cols = [c for c in ["Performance_Site", "Analyzer_Type", "n_results", "n_mrn", "Tests_per_MRN", "serial_inpatients", "mean_draws_serial", "raw_mean"] if c in s_sum.columns]

        img2_block = f'<div class="card"><h2>Visual Comparison of Sites by Analyzer: Distributions, `refineR` Intervals &amp; Diagnostic Sensitivity</h2><img class="fig-img" src="data:image/png;base64,{b64_2}" alt="Site Comparison Figure 2" /></div>' if b64_2 else ""
        img1_block = f'<div class="card"><h2>Regression Diagnostics: Pre- vs. Post-Exclusion Site Shifts &amp; Confounding Diagnosis Effects</h2><img class="fig-img" src="data:image/png;base64,{b64_1}" alt="Site Shifts Figure 1" /></div>' if b64_1 else ""

        html_content = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>ASCP 2026 — Multi-Site Calcium Reference Interval &amp; Harmonization Dashboard</title>
<style>
  :root {{ --navy:#0f172a; --blue:#1d4ed8; --orange:#ea580c; --green:#15803d; --bg:#f8fafc; --border:#e2e8f0; }}
  body {{ font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif; background: var(--bg); color: #1e293b; margin: 0; }}
  header {{ background: linear-gradient(135deg, #0f172a 0%, #1e3a8a 100%); color: white; padding: 22px 34px; display: flex; justify-content: space-between; align-items: center; flex-wrap: wrap; gap: 12px; }}
  header h1 {{ margin: 0; font-size: 21px; }}
  header p {{ margin: 5px 0 0 0; font-size: 13px; color: #cbd5e1; }}
  .badge {{ background: #38bdf8; color: #0f172a; font-weight: 700; font-size: 12px; padding: 6px 12px; border-radius: 6px; }}
  .kpi-strip {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(240px, 1fr)); gap: 14px; max-width: 1420px; margin: 20px auto 0 auto; padding: 0 26px; }}
  .kpi {{ background: white; border: 1px solid var(--border); border-left: 4px solid var(--blue); border-radius: 8px; padding: 14px 16px; }}
  .kpi .lbl {{ font-size: 11.5px; text-transform: uppercase; color: #64748b; font-weight: 700; }}
  .kpi .val {{ font-size: 20px; font-weight: 800; color: var(--navy); margin-top: 4px; }}
  .kpi .sub {{ font-size: 12px; color: #475569; margin-top: 2px; }}
  .nav-tabs {{ display: flex; flex-wrap: wrap; gap: 8px; background: white; padding: 12px 34px 0 34px; border-bottom: 2px solid var(--border); margin-top: 18px; position: sticky; top: 0; z-index: 10; }}
  .tab-btn {{ padding: 10px 18px; border: none; background: transparent; cursor: pointer; font-size: 13.5px; font-weight: 600; color: #64748b; border-bottom: 3px solid transparent; }}
  .tab-btn.active {{ color: var(--blue); border-bottom: 3px solid var(--blue); }}
  .container {{ max-width: 1420px; margin: 22px auto; padding: 0 26px 44px 26px; }}
  .tab-pane {{ display: none; }}
  .tab-pane.active {{ display: block; }}
  .card {{ background: white; border: 1px solid var(--border); border-radius: 10px; padding: 20px 22px; margin-bottom: 22px; box-shadow: 0 1px 3px rgba(0,0,0,0.04); }}
  .card h2 {{ margin-top: 0; font-size: 16.5px; color: var(--navy); border-bottom: 1px solid var(--border); padding-bottom: 9px; }}
  .grid-2 {{ display: grid; grid-template-columns: 1fr 1fr; gap: 20px; }}
  .fig-img {{ width: 100%; height: auto; border-radius: 6px; border: 1px solid var(--border); }}
  table.data-table {{ width: 100%; border-collapse: collapse; font-size: 12.8px; margin-top: 6px; }}
  table.data-table th {{ background: #f1f5f9; color: var(--navy); text-align: left; padding: 9px 11px; border-bottom: 2px solid #cbd5e1; }}
  table.data-table td {{ padding: 8px 11px; border-bottom: 1px solid var(--border); }}
  table.data-table tr:nth-child(even) {{ background: #f8fafc; }}
</style>
</head>
<body>
<header>
  <div>
    <h1>Applying R &amp; AI for Automated Reference Interval Review — Serum Calcium</h1>
    <p>4 Testing Sites &times; 2 Analyzer Platforms | Baseline Site: {LAB_REF} | Filter: BH-FDR q &lt; {FDR_THRESHOLD}, |Effect| &ge; {EFFECT_MIN_UNITS} mg/dL</p>
  </div>
  <div class="badge">ASCP 2026 QA Validation Report</div>
</header>

<div class="kpi-strip">
  <div class="kpi">
    <div class="lbl">1. Platform Analytical Bias</div>
    <div class="val">{omega_bias_val:+.2f} mg/dL</div>
    <div class="sub">Analyzer Omega vs. Alpha (p &lt; 0.0001)</div>
  </div>
  <div class="kpi" style="border-left-color:#ea580c;">
    <div class="lbl">2. Omega Overly Wide Mfg URL</div>
    <div class="val">8.3 – 10.1 mg/dL</div>
    <div class="sub">refineR RI vs. Mfg 8.4–10.5 (98.3% N=20 pass)</div>
  </div>
  <div class="kpi" style="border-left-color:#15803d;">
    <div class="lbl">3. E21.0 Hypercalcemia Sensitivity</div>
    <div class="val">37.0% &rarr; 66.0%</div>
    <div class="sub">Restored on Omega (matches Alpha 66.8%)</div>
  </div>
  <div class="kpi" style="border-left-color:#7c3aed;">
    <div class="lbl">4. Inpatient Posture / Fluid Shift</div>
    <div class="val">{inpt_shift_val:+.2f} mg/dL</div>
    <div class="sub">Multi-day repeat draws controlled via MRN</div>
  </div>
</div>

<div class="nav-tabs">
  <button class="tab-btn active" onclick="openTab('t1', this)">📊 1. Site Comparison &amp; Harmonization Graphs</button>
  <button class="tab-btn" onclick="openTab('t2', this)">⚖️ 2. refineR Intervals &amp; CLSI N=20 Blind Spot</button>
  <button class="tab-btn" onclick="openTab('t3', this)">🏥 3. Pre/Post Site Shifts &amp; 2&times;2 Breakdown</button>
  <button class="tab-btn" onclick="openTab('t4', this)">🚩 4. ICD-10 Confounder &amp; Phenotype Discovery</button>
</div>

<div class="container">
  <div id="t1" class="tab-pane active">
    {img2_block}
    {img1_block}
  </div>
  <div id="t2" class="tab-pane">
    <div class="card">
      <h2>Site-by-Site `refineR` Calcium Reference Intervals vs. Manufacturer Limits &amp; 20-Patient CLSI Pass Rate</h2>
      {_tbl_html(g["ri_per_site_df"][["Performance_Site", "Analyzer_Type", "n_clean", "Mfg_Stated_RI", "Raw_Unfiltered_95%", "refineR_Derived_RI", "Healthy_Inside_Mfg_%", "CLSI_N20_Pass_Rate_%", "Method"]])}
    </div>
    <div class="card">
      <h2>Clinical Harmonization Impact: Primary Hyperparathyroidism (`E21.0`) Detection Before vs. After `refineR` Review</h2>
      {_tbl_html(g["clinical_impact_df"])}
    </div>
  </div>
  <div id="t3" class="tab-pane">
    <div class="card">
      <h2>2&times;2 Breakdown on Clean Cohort (Analyzer Omega Bias vs. Inpatient Setting Shift)</h2>
      {_tbl_html(fact_df.round(4))}
    </div>
    <div class="card">
      <h2>Pre-Exclusion vs. Post-Exclusion Site Shifts (Referenced to {LAB_REF})</h2>
      {_tbl_html(pd.concat([g["site_effects_pre_df"], g["site_effects_post_df"]], ignore_index=True))}
    </div>
    <div class="grid-2">
      <div class="card">
        <h2>Pre-Analytical Serial Testing Summary by Site</h2>
        {_tbl_html(s_sum[site_sum_cols])}
      </div>
      <div class="card">
        <h2>Site Odds Ratios of Abnormal Flagging (Manufacturer RI)</h2>
        {_tbl_html(g["logit_site_df"])}
      </div>
    </div>
  </div>
  <div id="t4" class="tab-pane">
    <div class="grid-2">
      <div class="card">
        <h2>Cohort Filtration Summary</h2>
        {_tbl_html(g["cohort_sizes_df"])}
      </div>
      <div class="card">
        <h2>Unflagged Normal &amp; Routine Codes Retained (|Effect| &lt; {EFFECT_MIN_UNITS} mg/dL)</h2>
        {_tbl_html(g["unflagged_codes_df"][["ICD10_Code", "Clinical_Descriptor", "n_mrn", "Effect_mg_dL", "q_value"]].round(4))}
      </div>
    </div>
    <div class="card">
      <h2>Flagged Pathological &amp; Drug-Associated ICD-10 Codes Excluded (q &lt; {FDR_THRESHOLD} &amp; |Effect| &ge; {EFFECT_MIN_UNITS} mg/dL)</h2>
      {_tbl_html(g["sig_codes_df"].round(4))}
    </div>
  </div>
</div>

<script>
function openTab(id, btn) {{
  document.querySelectorAll('.tab-pane').forEach(p => p.classList.remove('active'));
  document.querySelectorAll('.tab-btn').forEach(b => b.classList.remove('active'));
  document.getElementById(id).classList.add('active');
  btn.classList.add('active');
}}
</script>
</body>
</html>"""

    with open(filename, "w", encoding="utf-8") as f:
        f.write(html_content)
    b64 = base64.b64encode(html_content.encode("utf-8")).decode("utf-8")
    display(HTML(f"""
    <div style="background:#0f172a; color:white; padding:18px 24px; border-radius:10px; margin:12px 0; display:flex; align-items:center; justify-content:space-between; flex-wrap:wrap; gap:12px; font-family:sans-serif;">
      <div>
        <div style="font-size:16px; font-weight:700; color:#38bdf8;">✅ Multi-Site Calcium QA Validation Dashboard Ready!</div>
        <div style="font-size:13px; color:#cbd5e1; margin-top:4px;">If your browser blocked the automatic new tab, click the button on the right to open the full interactive report.</div>
      </div>
      <a id="qa-tab-btn" target="_blank" style="background:#2563eb; color:white; padding:11px 20px; border-radius:7px; text-decoration:none; font-weight:700; font-size:14px; white-space:nowrap;">
        🚀 Open Report in New Web Tab
      </a>
    </div>
    <script>
      (function(){{
        const bin = atob("{b64}");
        const arr = new Uint8Array(bin.length);
        for (let i = 0; i < bin.length; i++) arr[i] = bin.charCodeAt(i);
        const url = URL.createObjectURL(new Blob([arr], {{type: "text/html;charset=utf-8"}}));
        const btn = document.getElementById("qa-tab-btn");
        if (btn) btn.href = url;
        window.open(url, "_blank");
      }})();
    </script>
    """))

# -------------------------------------------------------------------------
# Load or Generate Curated Multi-Site Calcium Dataset (`ascp_electrolyte_data.csv`)
# -------------------------------------------------------------------------
DATA_URL = "https://github.com/jtucci88/ascp-2026-refiner-workshop/raw/refs/heads/main/ascp_electrolyte_data.csv" #@param {type:"string"}

def generate_curated_calcium_csv(filepath="/content/ascp_electrolyte_data.csv"):
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    rng = np.random.default_rng(20261119)
    end_date = pd.Timestamp("2026-11-19")
    start_date = pd.Timestamp("2026-05-20")
    total_days = (end_date - start_date).days

    sites_config = [
        {"site": "Downtown Inpatient Hospital and Onsite Clinics", "analyzer": "Analyzer Alpha", "mfg_ri": "8.6-10.4", "target_rows": 7600, "inpt_frac": 0.46, "bias": 0.00},
        {"site": "Downtown Outpatient Clinic",                     "analyzer": "Analyzer Omega", "mfg_ri": "8.4-10.5", "target_rows": 6000, "inpt_frac": 0.00, "bias": -0.30},
        {"site": "Regional Inpatient Hospital and Onsite Clinics", "analyzer": "Analyzer Omega", "mfg_ri": "8.4-10.5", "target_rows": 6400, "inpt_frac": 0.44, "bias": -0.30},
        {"site": "Regional Outpatient Clinic",                     "analyzer": "Analyzer Alpha", "mfg_ri": "8.6-10.4", "target_rows": 6000, "inpt_frac": 0.00, "bias": 0.00},
    ]

    icd_catalog = [
        ("Z00.00", 0.245, 0.055,  0.00, 0.00), ("Z12.31", 0.105, 0.015,  0.00, 0.00),
        ("Z12.11", 0.090, 0.015,  0.00, 0.00), ("Z12.5",  0.080, 0.015,  0.00, 0.00),
        ("Z01.818",0.065, 0.110,  0.00, 0.00), ("Z23",    0.045, 0.010,  0.00, 0.00),
        ("I10",    0.085, 0.120,  0.01, 0.00), ("E78.5",  0.065, 0.075,  0.01, 0.00),
        ("E11.9",  0.045, 0.070, -0.02, 0.00), ("R07.9",  0.015, 0.085,  0.00, 0.00),
        ("M17.11", 0.020, 0.060,  0.00, 0.00),
        ("E55.9",  0.028, 0.035, -0.30, 0.08), ("N18.4",  0.014, 0.050, -0.46, 0.14),
        ("N18.6",  0.010, 0.065, -0.72, 0.20), ("E88.09", 0.008, 0.068, -0.86, 0.18),
        ("A41.9",  0.005, 0.065, -0.78, 0.20), ("K85.90", 0.004, 0.032, -0.96, 0.22),
        ("E21.0",  0.022, 0.012, +1.18, 0.24), ("F31.9",  0.018, 0.012, +0.36, 0.10),
        ("D86.9",  0.012, 0.008, +0.68, 0.18), ("C90.00", 0.010, 0.012, +1.12, 0.26),
        ("C79.51", 0.009, 0.011, +1.36, 0.28),
    ]
    outpt_p = np.array([c[1] for c in icd_catalog], dtype=float); outpt_p /= outpt_p.sum()
    inpt_p  = np.array([c[2] for c in icd_catalog], dtype=float); inpt_p  /= inpt_p.sum()
    sec_pool = ["I10", "E78.5", "Z00.00", "E11.9"]

    rows, mrn_seq, spec_seq = [], 100001, 5000001
    for sc in sites_config:
        cnt = 0
        while cnt < sc["target_rows"]:
            mrn = f"MRN{mrn_seq:06d}"; mrn_seq += 1
            is_inpt = rng.random() < sc["inpt_frac"]
            if is_inpt:
                age = int(np.clip(np.round(rng.normal(61, 16)), 18, 94))
                n_draws = int(rng.choice([1, 2, 3, 4, 5, 6], p=[0.26, 0.28, 0.21, 0.13, 0.08, 0.04]))
                code, _, _, sh, esd = icd_catalog[int(rng.choice(len(icd_catalog), p=inpt_p))]
                posture = -0.14
            else:
                age = int(np.clip(np.round(rng.normal(50, 17)), 18, 92))
                n_draws = 1
                code, _, _, sh, esd = icd_catalog[int(rng.choice(len(icd_catalog), p=outpt_p))]
                posture = 0.0

            icd_str = code
            if rng.random() < 0.28:
                sec = str(rng.choice(sec_pool))
                if sec != code:
                    icd_str = f"{code}; {sec}"

            setpoint = 9.50 + sc["bias"] + posture + sh + rng.normal(0.0, np.sqrt(0.35**2 + esd**2))
            max_span = 12 if n_draws > 1 else 0
            d0 = int(rng.integers(0, total_days - max_span + 1))
            offsets = [d0] if n_draws == 1 else [d0] + [d0 + d for d in sorted(rng.choice(np.arange(1, max_span + 1), size=n_draws - 1, replace=False).tolist())]

            for d_off in offsets:
                val = float(np.clip(np.round(setpoint + rng.normal(0.0, 0.28), 1), 5.2, 14.8))
                rows.append({
                    "MRN": mrn,
                    "Specimen_ID": f"SPEC{spec_seq:07d}",
                    "Patient_Age": age,
                    "Date_of_Sample": (start_date + pd.Timedelta(days=int(d_off))).strftime("%Y-%m-%d"),
                    "Performance_Site": sc["site"],
                    "Analyzer_Type": sc["analyzer"],
                    "ICD10_Codes": icd_str,
                    "Analyte_Name": "Calcium",
                    "Result_Value": val,
                    "Result_Units": "mg/dL",
                    "Manufacturer_Stated_RI": sc["mfg_ri"],
                })
                spec_seq += 1; cnt += 1

    df_gen = pd.DataFrame(rows).sort_values(["Date_of_Sample", "Specimen_ID"]).reset_index(drop=True)
    df_gen["Specimen_ID"] = [f"SPEC{5000001 + i:07d}" for i in range(len(df_gen))]
    df_gen.to_csv(filepath, index=False)
    return df_gen

print("⏳ [2/3] Loading `ascp_electrolyte_data.csv` into isolated Colab VM...")
CSV_PATH = "/content/ascp_electrolyte_data.csv"
os.makedirs("/content", exist_ok=True)

if DATA_URL.strip():
    clean_url = DATA_URL.strip()
    if "github.com/" in clean_url and "/blob/" in clean_url:
        clean_url = clean_url.replace("github.com/", "raw.githubusercontent.com/").replace("/blob/", "/")
    try:
        df_raw = pd.read_csv(clean_url)
        df_raw.to_csv(CSV_PATH, index=False)
        print(f"✅ Downloaded dataset from GitHub DATA_URL ({len(df_raw):,} rows)!")
    except Exception as url_err:
        alt_url = clean_url.replace("github.com/", "raw.githubusercontent.com/").replace("/raw/refs/heads/", "/refs/heads/").replace("/raw/", "/")
        try:
            df_raw = pd.read_csv(alt_url)
            df_raw.to_csv(CSV_PATH, index=False)
            print(f"✅ Downloaded dataset from GitHub Raw mirror ({len(df_raw):,} rows)!")
        except Exception:
            print(f"⚠️ Could not reach DATA_URL ({url_err}); building exact curated dataset locally...")
            df_raw = generate_curated_calcium_csv(CSV_PATH)
elif os.path.exists(CSV_PATH):
    df_raw = pd.read_csv(CSV_PATH)
else:
    df_raw = generate_curated_calcium_csv(CSV_PATH)

# Default Guardrail Configuration Variables (adjustable in Step 0B below)
LAB_REF = "Regional Outpatient Clinic"
CODE_MIN_MRN = 150
FDR_THRESHOLD = 0.05
EFFECT_MIN_UNITS = 0.10
EXCLUSION_POLICY = "strict"
DEDUPLICATE_FIRST_MRN_FOR_REFINER = False
MFG_RI_MAP = {"Analyzer Alpha": (8.6, 10.4), "Analyzer Omega": (8.4, 10.5)}
REF_RI_WIDTH = 1.8

print(f"✅ [3/3] Setup Complete! Loaded {len(df_raw):,} Calcium results ({df_raw['MRN'].nunique():,} unique MRNs) at `{CSV_PATH}`.")


In [ ]:
#@title ⚙️ Step 0B: Pipeline Configuration Guardrails (Optional — Adjust Statistical Thresholds Safely) { display-mode: "form" }
LAB_REF = "Regional Outpatient Clinic" #@param ["Regional Outpatient Clinic", "Downtown Inpatient Hospital and Onsite Clinics", "Downtown Outpatient Clinic", "Regional Inpatient Hospital and Onsite Clinics"]
CODE_MIN_MRN = 150 #@param {type:"slider", min:50, max:400, step:25}
FDR_THRESHOLD = 0.05 #@param {type:"number"}
EFFECT_MIN_UNITS = 0.10 #@param {type:"number"}
# EXCLUSION_POLICY: "strict" excludes any order with >=1 flagged code; "lenient" tolerates up to 1 flagged code (excludes >=2).
EXCLUSION_POLICY = "strict" #@param ["strict", "lenient"]
DEDUPLICATE_FIRST_MRN_FOR_REFINER = False #@param {type:"boolean"}

MFG_RI_MAP = {
    "Analyzer Alpha": (8.6, 10.4),  # Width = 1.8 mg/dL
    "Analyzer Omega": (8.4, 10.5),  # Width = 2.1 mg/dL
}
REF_RI_WIDTH = 1.8  # Baseline Analyzer Alpha RI width (10.4 - 8.6 = 1.8 mg/dL)

policy_desc = "0 flagged codes allowed (excludes >=1)" if EXCLUSION_POLICY == "strict" else "up to 1 flagged code allowed (excludes >=2)"
print("✅ Active Pipeline Guardrails:")
print(f"   • Baseline Reference Site : {LAB_REF} (Baseline RI Width = {REF_RI_WIDTH} mg/dL)")
print(f"   • Min Unique MRNs / Code  : {CODE_MIN_MRN} patients")
print(f"   • Confounder Filter Rule  : BH-FDR q < {FDR_THRESHOLD} AND |Effect| >= {EFFECT_MIN_UNITS} mg/dL")
print(f"   • Exclusion Policy        : {EXCLUSION_POLICY} — {policy_desc}")
print(f"   • Deduplicate Serial MRN  : {DEDUPLICATE_FIRST_MRN_FOR_REFINER} (Toggle True to restrict refineR to 1st draw/patient)")


## 🧪 Walkthrough Step 1: Review the 6-Month LIS Calcium Extract & Inpatient Repeat Draws

In [ ]:
#@title 📋 Step 1 Prompt Card (Wrapped Text + One-Click Copy) { display-mode: "form" }
import html as _html
from IPython.display import HTML as _HTML, display as _display

_prompt_text = """Load our 6-month LIS calcium extract (/content/ascp_electrolyte_data.csv) to prepare it for a multi-site reference interval review. Clean up any extra spaces in the text columns, confirm that only adult patients (age 18 and older) are included, and separate multiple semicolon-separated ICD10_Codes on the same order into a list column named icd_list.

Then create a site summary table (site_summary_df) grouped by Performance_Site, Analyzer_Type, and Manufacturer_Stated_RI showing:
- Total calcium results and unique patients (MRN)
- Average tests per patient (Tests_per_MRN) and how many patients had repeat draws across different calendar days
- Raw calcium mean, SD, and central 95% range (2.5th to 97.5th percentiles)

Save the cleaned adult dataset as df_adult and display site_summary_df."""
_escaped_label = _html.escape('🧪 Step 1 Prompt')
_escaped_body = _html.escape(_prompt_text)

_display(_HTML(f"""<div style="border:1px solid #cbd5e1; border-left:5px solid #2563eb; border-radius:8px; background:#f8fafc; padding:14px 18px; margin:8px 0; font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:12px; margin-bottom:10px;">
    <span style="font-weight:700; font-size:14px; color:#0f172a;">{_escaped_label} &mdash; Click Copy, then click &ldquo;generate&rdquo; in the empty code cell below:</span>
    <button onclick="(function(btn){{
      const el = document.getElementById('p_step1');
      const txt = el ? el.innerText : '';
      const ta = document.createElement('textarea');
      ta.value = txt;
      ta.setAttribute('readonly', '');
      ta.style.position = 'fixed';
      ta.style.top = '-9999px';
      document.body.appendChild(ta);
      ta.focus();
      ta.select();
      let ok = false;
      try {{ ok = document.execCommand('copy'); }} catch(e) {{}}
      document.body.removeChild(ta);
      if (!ok && navigator.clipboard) {{ navigator.clipboard.writeText(txt); }}
      btn.innerText = '✅ Copied!';
      btn.style.background = '#16a34a';
      setTimeout(function(){{
        btn.innerText = '📋 Copy Prompt';
        btn.style.background = '#2563eb';
      }}, 1800);
    }})(this)" style="background:#2563eb; color:#ffffff; border:none; border-radius:6px; padding:7px 14px; font-size:13px; font-weight:600; cursor:pointer; white-space:nowrap; box-shadow:0 1px 2px rgba(0,0,0,0.12);">📋 Copy Prompt</button>
  </div>
  <div id="p_step1" style="white-space:pre-wrap; word-wrap:break-word; overflow-wrap:anywhere; background:#ffffff; border:1px solid #e2e8f0; border-radius:6px; padding:12px 14px; font-family:ui-monospace,SFMono-Regular,Consolas,monospace; font-size:13px; line-height:1.55; color:#1e293b;">{_escaped_body}</div>
</div>"""))


In [ ]:
#@title 🛡️ [Guardrail Checkpoint 1] Click Play to Sync `df_adult` & Inspect Serial Inpatient Draws { display-mode: "form" }
if "_sync_step1" not in globals() or not os.path.exists("/content/ascp_electrolyte_data.csv"):
    raise RuntimeError("Please click Play on 🚀 Step 0 at the top of the notebook first!")

_sync_step1(force=True)

print(f"✅ Step 1 Synced: {len(df_adult):,} adult Calcium results ({df_adult['MRN'].nunique():,} MRNs) from {df_adult['Date_of_Sample'].min()} to {df_adult['Date_of_Sample'].max()}.")
print("🔍 CLINICAL QA CHECK:")
print("   • Raw Acuity Illusion: Downtown Inpatient (Analyzer Alpha, raw mean 9.31) looks deceptively close to Downtown Outpatient (Analyzer Omega, raw mean 9.23)")
print("     because hospitalized hypocalcemia pulls Downtown Alpha downward, masking Analyzer Omega's negative analytical bias.")
print("   • Inpatient Serial Draws: ~1.69–1.77 tests/patient at hospital sites (up to 6 distinct days) vs. 1.00 at outpatient clinics.")
from IPython.display import display
display(site_summary_df[[
    "Performance_Site", "Analyzer_Type", "Manufacturer_Stated_RI",
    "n_results", "n_mrn", "Tests_per_MRN", "serial_inpatients", "raw_mean", "raw_sd", "raw_p2_5", "raw_p97_5"
]])


## 📉 Walkthrough Step 2: Separate Sick/Inpatient Confounders from True Analyzer Bias

In [ ]:
#@title 📋 Step 2 Prompt Card (Wrapped Text + One-Click Copy) { display-mode: "form" }
import html as _html
from IPython.display import HTML as _HTML, display as _display

_prompt_text = """Using df_adult and the guardrail settings loaded in our workspace (with LAB_REF as our baseline outpatient comparison site), help us separate clinical illness from true instrument differences:

1. Compare raw calcium differences across all 4 testing sites against LAB_REF (both in mg/dL and as a % of our 1.8 mg/dL baseline reference interval width), accounting for repeat draws from the same MRN so hospitalized patients with serial labs don't skew the statistics (site_effects_pre_df).
2. Screen every diagnosis code in icd_list (with at least CODE_MIN_MRN unique patients) to measure how much each condition raises or lowers calcium after accounting for testing site. Flag confounding diagnoses that remain statistically significant after false-positive adjustment (FDR_THRESHOLD) AND shift calcium by at least EFFECT_MIN_UNITS (0.10 mg/dL), using ICD_DESCRIPTORS to label each condition (sig_codes_df). Exclude orders with flagged codes to create our clean reference population (df_clean).
3. Re-check the site differences on df_clean (site_effects_post_df), and break down how much of the remaining shift comes from Analyzer Omega vs. Analyzer Alpha versus an Inpatient vs. Outpatient hospital setting (factorial_2x2_df).
4. Calculate each site's odds ratio of flagging abnormal against Manufacturer_Stated_RI (logit_site_df). Display our clean cohort counts, the flagged diagnosis codes, and the 2x2 analyzer vs. inpatient breakdown."""
_escaped_label = _html.escape('📉 Step 2 Prompt')
_escaped_body = _html.escape(_prompt_text)

_display(_HTML(f"""<div style="border:1px solid #cbd5e1; border-left:5px solid #2563eb; border-radius:8px; background:#f8fafc; padding:14px 18px; margin:8px 0; font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:12px; margin-bottom:10px;">
    <span style="font-weight:700; font-size:14px; color:#0f172a;">{_escaped_label} &mdash; Click Copy, then click &ldquo;generate&rdquo; in the empty code cell below:</span>
    <button onclick="(function(btn){{
      const el = document.getElementById('p_step2');
      const txt = el ? el.innerText : '';
      const ta = document.createElement('textarea');
      ta.value = txt;
      ta.setAttribute('readonly', '');
      ta.style.position = 'fixed';
      ta.style.top = '-9999px';
      document.body.appendChild(ta);
      ta.focus();
      ta.select();
      let ok = false;
      try {{ ok = document.execCommand('copy'); }} catch(e) {{}}
      document.body.removeChild(ta);
      if (!ok && navigator.clipboard) {{ navigator.clipboard.writeText(txt); }}
      btn.innerText = '✅ Copied!';
      btn.style.background = '#16a34a';
      setTimeout(function(){{
        btn.innerText = '📋 Copy Prompt';
        btn.style.background = '#2563eb';
      }}, 1800);
    }})(this)" style="background:#2563eb; color:#ffffff; border:none; border-radius:6px; padding:7px 14px; font-size:13px; font-weight:600; cursor:pointer; white-space:nowrap; box-shadow:0 1px 2px rgba(0,0,0,0.12);">📋 Copy Prompt</button>
  </div>
  <div id="p_step2" style="white-space:pre-wrap; word-wrap:break-word; overflow-wrap:anywhere; background:#ffffff; border:1px solid #e2e8f0; border-radius:6px; padding:12px 14px; font-family:ui-monospace,SFMono-Regular,Consolas,monospace; font-size:13px; line-height:1.55; color:#1e293b;">{_escaped_body}</div>
</div>"""))


In [ ]:
#@title 🛡️ [Guardrail Checkpoint 2] Click Play to Sync ICD-10 Confounder Exclusions, 2×2 Breakdown & `df_clean` { display-mode: "form" }
if "_sync_step2" not in globals() or not os.path.exists("/content/ascp_electrolyte_data.csv"):
    raise RuntimeError("Please click Play on 🚀 Step 0 at the top of the notebook first!")

_sync_step2(force=True)

from IPython.display import display
print(f"✅ Step 2 Synced! Retained {len(df_clean):,} clean reference Calcium results across {df_clean['MRN'].nunique():,} patients.")
print("\n🔬 2×2 Analyzer vs. Inpatient Setting Breakdown on Clean Cohort:")
display(factorial_2x2_df[["term", "coef", "%_of_RI_Width", "std_err", "p_value", "ci_lower_95", "ci_upper_95"]].round(4))
print("\n🚩 11 Flagged Confounding Diagnosis Codes Excluded (Including `E21.0` Hyperparathyroidism & `F31.9` Bipolar/Lithium shift):")
display(sig_codes_df[["ICD10_Code", "Clinical_Descriptor", "n_mrn", "Effect_mg_dL", "%_RI_Width", "q_value"]])
print(f"\n✅ 11 Routine / Unbiased Codes Retained (|Effect| < {EFFECT_MIN_UNITS} mg/dL protects healthy screening encounters):")
display(unflagged_codes_df[["ICD10_Code", "Clinical_Descriptor", "n_mrn", "Effect_mg_dL", "q_value"]].head(6))


## ⚖️ Walkthrough Step 3: Derive `refineR` Intervals & Test 20-Patient CLSI Verification

In [ ]:
#@title 📋 Step 3 Prompt Card (Wrapped Text + One-Click Copy) { display-mode: "form" }
import html as _html
from IPython.display import HTML as _HTML, display as _display

_prompt_text = """Using our clean reference population (df_clean), run the R refineR package to calculate indirect 2.5th to 97.5th percentile calcium reference intervals (rounded to 1 decimal place) for each of the 4 Performance_Site locations and both analyzers. Build a site comparison table (ri_per_site_df) placing each site's refineR interval next to its Manufacturer_Stated_RI (8.6-10.4 mg/dL for Analyzer Alpha; 8.4-10.5 mg/dL for Analyzer Omega) and raw unfiltered 95% range.

Next, test why routine small-sample verifications can be misleading:
1. For each site, simulate 1,000 standard CLSI 20-patient verification studies by randomly pulling 20 results from df_clean at a time and recording what % of the time at least 18 of 20 results fall inside Manufacturer_Stated_RI.
2. Pull all patients with Primary Hyperparathyroidism (E21.0) from df_adult and build a summary table (clinical_impact_df) comparing what % of cases are flagged high on Analyzer Alpha and Analyzer Omega using the manufacturer upper limit versus our refineR upper limit.

Display ri_per_site_df and clinical_impact_df."""
_escaped_label = _html.escape('⚖️ Step 3 Prompt')
_escaped_body = _html.escape(_prompt_text)

_display(_HTML(f"""<div style="border:1px solid #cbd5e1; border-left:5px solid #2563eb; border-radius:8px; background:#f8fafc; padding:14px 18px; margin:8px 0; font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:12px; margin-bottom:10px;">
    <span style="font-weight:700; font-size:14px; color:#0f172a;">{_escaped_label} &mdash; Click Copy, then click &ldquo;generate&rdquo; in the empty code cell below:</span>
    <button onclick="(function(btn){{
      const el = document.getElementById('p_step3');
      const txt = el ? el.innerText : '';
      const ta = document.createElement('textarea');
      ta.value = txt;
      ta.setAttribute('readonly', '');
      ta.style.position = 'fixed';
      ta.style.top = '-9999px';
      document.body.appendChild(ta);
      ta.focus();
      ta.select();
      let ok = false;
      try {{ ok = document.execCommand('copy'); }} catch(e) {{}}
      document.body.removeChild(ta);
      if (!ok && navigator.clipboard) {{ navigator.clipboard.writeText(txt); }}
      btn.innerText = '✅ Copied!';
      btn.style.background = '#16a34a';
      setTimeout(function(){{
        btn.innerText = '📋 Copy Prompt';
        btn.style.background = '#2563eb';
      }}, 1800);
    }})(this)" style="background:#2563eb; color:#ffffff; border:none; border-radius:6px; padding:7px 14px; font-size:13px; font-weight:600; cursor:pointer; white-space:nowrap; box-shadow:0 1px 2px rgba(0,0,0,0.12);">📋 Copy Prompt</button>
  </div>
  <div id="p_step3" style="white-space:pre-wrap; word-wrap:break-word; overflow-wrap:anywhere; background:#ffffff; border:1px solid #e2e8f0; border-radius:6px; padding:12px 14px; font-family:ui-monospace,SFMono-Regular,Consolas,monospace; font-size:13px; line-height:1.55; color:#1e293b;">{_escaped_body}</div>
</div>"""))


In [ ]:
#@title 🛡️ [Guardrail Checkpoint 3] Click Play to Run `refineR` & Simulate 20-Patient CLSI Verification Blind Spot { display-mode: "form" }
if "_sync_step3" not in globals() or not os.path.exists("/content/ascp_electrolyte_data.csv"):
    raise RuntimeError("Please click Play on 🚀 Step 0 at the top of the notebook first!")

print("⏳ Running `refineR` inverse Box-Cox modeling across all 4 testing sites and both analyzer platforms...")
_sync_step3(force=True)

from IPython.display import display
print("✅ Step 3 Synced! Site-by-Site `refineR` Calcium Intervals vs. Manufacturer Stated RIs:")
display(ri_per_site_df[[
    "Performance_Site", "Analyzer_Type", "n_clean", "Mfg_Stated_RI",
    "Raw_Unfiltered_95%", "refineR_Derived_RI", "Healthy_Inside_Mfg_%", "CLSI_N20_Pass_Rate_%"
]])
print("\n🚨 CLINICAL IMPACT — Detection of Primary Hyperparathyroidism (`E21.0`) Before vs. After `refineR` Review:")
display(clinical_impact_df)


## 📊 Walkthrough Step 4: Build Visual QA Validation Packet & Launch New-Tab Dashboard

In [ ]:
#@title 📋 Step 4 Prompt Card (Wrapped Text + One-Click Copy) { display-mode: "form" }
import html as _html
from IPython.display import HTML as _HTML, display as _display

_prompt_text = """Write concise Python matplotlib/seaborn code (~65 lines total; do NOT write raw HTML strings) to create two 2-panel QA figures comparing our 4 sites and 2 analyzers:

1. fig1 (1x2 subplots):
   - Left: Plot each site's calcium shift vs. LAB_REF before (site_effects_pre_df) vs. after (site_effects_post_df) ICD-10 filtering with 95% CIs.
   - Right: Forest plot of excluded confounding ICD-10 shifts (sig_codes_df, highlighting E21.0 and F31.9) alongside retained routine codes (Z00.00, Z12.31, I10).
2. fig2 (1x2 subplots):
   - Left: Violin/boxplots of clean calcium results (df_clean) across all 4 sites colored by Analyzer_Type, marking manufacturer limits (dashed lines), refineR limits (solid lines from ri_per_site_df), and shading the 10.1-10.5 mg/dL missed hypercalcemia zone on Analyzer Omega.
   - Right: Grouped bar chart comparing each analyzer's 20-patient CLSI verification pass rate against E21.0 hypercalcemia sensitivity using manufacturer vs. refineR limits (clinical_impact_df).

Call launch_html_dashboard(fig1, fig2) to package both figures and all summary tables into an interactive multi-tab HTML report in a new browser tab, then display fig2 and fig1 inline."""
_escaped_label = _html.escape('📊 Step 4 Prompt')
_escaped_body = _html.escape(_prompt_text)

_display(_HTML(f"""<div style="border:1px solid #cbd5e1; border-left:5px solid #2563eb; border-radius:8px; background:#f8fafc; padding:14px 18px; margin:8px 0; font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:12px; margin-bottom:10px;">
    <span style="font-weight:700; font-size:14px; color:#0f172a;">{_escaped_label} &mdash; Click Copy, then click &ldquo;generate&rdquo; in the empty code cell below:</span>
    <button onclick="(function(btn){{
      const el = document.getElementById('p_step4');
      const txt = el ? el.innerText : '';
      const ta = document.createElement('textarea');
      ta.value = txt;
      ta.setAttribute('readonly', '');
      ta.style.position = 'fixed';
      ta.style.top = '-9999px';
      document.body.appendChild(ta);
      ta.focus();
      ta.select();
      let ok = false;
      try {{ ok = document.execCommand('copy'); }} catch(e) {{}}
      document.body.removeChild(ta);
      if (!ok && navigator.clipboard) {{ navigator.clipboard.writeText(txt); }}
      btn.innerText = '✅ Copied!';
      btn.style.background = '#16a34a';
      setTimeout(function(){{
        btn.innerText = '📋 Copy Prompt';
        btn.style.background = '#2563eb';
      }}, 1800);
    }})(this)" style="background:#2563eb; color:#ffffff; border:none; border-radius:6px; padding:7px 14px; font-size:13px; font-weight:600; cursor:pointer; white-space:nowrap; box-shadow:0 1px 2px rgba(0,0,0,0.12);">📋 Copy Prompt</button>
  </div>
  <div id="p_step4" style="white-space:pre-wrap; word-wrap:break-word; overflow-wrap:anywhere; background:#ffffff; border:1px solid #e2e8f0; border-radius:6px; padding:12px 14px; font-family:ui-monospace,SFMono-Regular,Consolas,monospace; font-size:13px; line-height:1.55; color:#1e293b;">{_escaped_body}</div>
</div>"""))


In [ ]:
#@title 🛡️ [Guardrail Checkpoint 4] Click Play to Render Site Comparison Graphs & Launch Dashboard in New Web Tab { display-mode: "form" }
from IPython.display import Image, display

if "_sync_step3" not in globals() or not os.path.exists("/content/ascp_electrolyte_data.csv"):
    raise RuntimeError("Please click Play on 🚀 Step 0 at the top of the notebook first!")

# Automatically validate & backfill canonical schemas if Gemini in Steps 1–3 used custom column names!
_sync_step3(force=False)

order_4_sites = [
    "Regional Outpatient Clinic",
    "Downtown Inpatient Hospital and Onsite Clinics",
    "Downtown Outpatient Clinic",
    "Regional Inpatient Hospital and Onsite Clinics",
]
short_site_labels = [
    "Regional Outpatient\n(Analyzer Alpha)",
    "Downtown Inpatient+Clinics\n(Analyzer Alpha)",
    "Downtown Outpatient\n(Analyzer Omega)",
    "Regional Inpatient+Clinics\n(Analyzer Omega)",
]

# -------------------------------------------------------------------------
# FIGURE 1: Pre vs. Post Site Shifts (Acuity Illusion) & ICD-10 Confounder Forest Plot
# -------------------------------------------------------------------------
fig1, (ax1a, ax1b) = plt.subplots(1, 2, figsize=(15.5, 6.2), dpi=130)

y_base = np.arange(len(order_4_sites))
pre_indexed = site_effects_pre_df.set_index("Performance_Site").loc[order_4_sites]
post_indexed = site_effects_post_df.set_index("Performance_Site").loc[order_4_sites]

ax1a.errorbar(
    pre_indexed["Shift_mg_dL"], y_base - 0.14,
    xerr=[pre_indexed["Shift_mg_dL"] - pre_indexed["CI_95_Low"], pre_indexed["CI_95_High"] - pre_indexed["Shift_mg_dL"]],
    fmt="s", color="#94a3b8", ecolor="#94a3b8", elinewidth=2, capsize=4, markersize=7,
    label="Pre-Exclusion Raw Shift (Confounded by Inpatient Acuity)"
)
ax1a.errorbar(
    post_indexed["Shift_mg_dL"], y_base + 0.14,
    xerr=[post_indexed["Shift_mg_dL"] - post_indexed["CI_95_Low"], post_indexed["CI_95_High"] - post_indexed["Shift_mg_dL"]],
    fmt="o", color="#1d4ed8", ecolor="#1d4ed8", elinewidth=2.4, capsize=4, markersize=8,
    label="Post-Exclusion Shift (True Platform + Posture Effect)"
)
for i, s in enumerate(order_4_sites):
    val_post = post_indexed.loc[s, "Shift_mg_dL"]
    pct_post = post_indexed.loc[s, "% Effect Size"]
    ax1a.text(
        val_post - 0.02 if val_post < 0 else val_post + 0.02, i + 0.28,
        f"{val_post:+.2f} mg/dL ({pct_post:+.1f}% RI)",
        fontsize=8.5, fontweight="bold", color="#1e3a8a",
        ha="right" if val_post < 0 else "left"
    )

ax1a.axvline(0.0, color="#0f172a", linestyle="--", linewidth=1.3)
ax1a.set_yticks(y_base)
ax1a.set_yticklabels(short_site_labels, fontsize=9, fontweight="bold")
ax1a.invert_yaxis()
ax1a.set_xlim(-0.62, 0.18)
ax1a.set_xlabel(f"Calcium Shift vs. {LAB_REF} (mg/dL, MRN-Clustered 95% CI)", fontsize=9.5, fontweight="bold")
ax1a.set_title("A. Unmasking Platform Bias vs. Raw Inpatient Acuity Illusion", fontsize=11, fontweight="bold")
ax1a.legend(loc="lower left", fontsize=8.5, frameon=True)
ax1a.grid(axis="x", linestyle=":", alpha=0.6)

plot_codes = pd.concat([
    sig_codes_df.head(6),
    unflagged_codes_df[unflagged_codes_df["ICD10_Code"].isin(["Z00.00", "Z12.31", "I10"])],
    sig_codes_df.tail(5)
], ignore_index=True).drop_duplicates("ICD10_Code").sort_values("Effect_mg_dL")

y_c = np.arange(len(plot_codes))
c_colors = [
    "#dc2626" if (r["Flagged_Confounder"] and r["Effect_mg_dL"] > 0)
    else ("#2563eb" if r["Flagged_Confounder"] else "#16a34a")
    for _, r in plot_codes.iterrows()
]
for idx_c, (_, r) in enumerate(plot_codes.iterrows()):
    ax1b.errorbar(
        r["Effect_mg_dL"], idx_c,
        xerr=[[r["Effect_mg_dL"] - r["CI_Low"]], [r["CI_High"] - r["Effect_mg_dL"]]],
        fmt="o", color=c_colors[idx_c], ecolor=c_colors[idx_c], elinewidth=2, capsize=3, markersize=6.5
    )
ax1b.axvline(0.0, color="#0f172a", linestyle="--", linewidth=1.1)
ax1b.axvspan(-EFFECT_MIN_UNITS, EFFECT_MIN_UNITS, color="#bbf7d0", alpha=0.35, label=f"Equivalence Zone (±{EFFECT_MIN_UNITS} mg/dL)")
ax1b.set_yticks(y_c)
ax1b.set_yticklabels([f"{r['ICD10_Code']} ({r['Clinical_Descriptor'].split('(')[0].strip()[:28]})" for _, r in plot_codes.iterrows()], fontsize=8.3)
ax1b.set_xlabel("Incremental Calcium Shift Controlling for Site (mg/dL)", fontsize=9.5, fontweight="bold")
ax1b.set_title("B. Automated ICD-10 Confounder Discovery (Red/Blue Excluded; Green Retained)", fontsize=11, fontweight="bold")
ax1b.legend(loc="lower right", fontsize=8.5)
ax1b.grid(axis="x", linestyle=":", alpha=0.6)

plt.tight_layout()

# -------------------------------------------------------------------------
# FIGURE 2: Multi-Site Calcium Distributions, refineR vs. Mfg RIs & CLSI Blind Spot
# -------------------------------------------------------------------------
fig2, (ax2a, ax2b) = plt.subplots(1, 2, figsize=(15.5, 6.2), dpi=130, gridspec_kw={"width_ratios": [1.35, 1.0]})

sns.violinplot(
    data=df_clean, x="Performance_Site", y="Result_Value",
    order=order_4_sites, hue="Analyzer_Type", dodge=False,
    palette={"Analyzer Alpha": "#2563eb", "Analyzer Omega": "#ea580c"},
    inner="box", cut=0, alpha=0.78, ax=ax2a
)

ri_indexed = ri_per_site_df.set_index("Performance_Site").loc[order_4_sites]
for i, s in enumerate(order_4_sites):
    r_lo, r_hi = ri_indexed.loc[s, "refineR_Low"], ri_indexed.loc[s, "refineR_High"]
    m_lo, m_hi = ri_indexed.loc[s, "Mfg_Low"], ri_indexed.loc[s, "Mfg_High"]
    ax2a.hlines([m_lo, m_hi], i - 0.34, i + 0.34, colors="#334155", linestyles="--", linewidth=1.7,
                label="Manufacturer Stated RI" if i == 0 else "")
    ax2a.hlines([r_lo, r_hi], i - 0.26, i + 0.26, colors="#15803d", linestyles="-", linewidth=2.8,
                label="refineR Derived RI (2.5%–97.5%)" if i == 0 else "")
    if "Omega" in ri_indexed.loc[s, "Analyzer_Type"]:
        ax2a.fill_between([i - 0.34, i + 0.34], r_hi, m_hi, color="#fde047", alpha=0.45,
                          label="Missed Mild Hypercalcemia Zone (Omega 10.1–10.5)" if i == 2 else "")

ax2a.set_xticks(range(len(order_4_sites)))
ax2a.set_xticklabels(short_site_labels, fontsize=8.5, fontweight="bold")
ax2a.set_ylim(7.5, 11.2)
ax2a.set_xlabel("")
ax2a.set_ylabel("Serum Calcium (mg/dL) — Clean Reference Cohort", fontsize=9.5, fontweight="bold")
ax2a.set_title("A. Comparison of Sites by Analyzer: `refineR` Intervals vs. Manufacturer Limits", fontsize=11, fontweight="bold")
ax2a.legend(loc="upper right", fontsize=8, framealpha=0.92)
ax2a.grid(axis="y", linestyle=":", alpha=0.6)

x_idx = np.arange(2)
width_b = 0.25
alpha_clsi = ri_indexed.loc["Regional Outpatient Clinic", "CLSI_N20_Pass_Rate_%"]
omega_clsi = ri_indexed.loc["Downtown Outpatient Clinic", "CLSI_N20_Pass_Rate_%"]
sens_mfg_vals = clinical_impact_df["Sensitivity_Using_Mfg_RI_%"].values
sens_ref_vals = clinical_impact_df["Sensitivity_Using_refineR_RI_%"].values

b1 = ax2b.bar(x_idx - width_b, [alpha_clsi, omega_clsi], width_b, color="#64748b", label="CLSI N=20 Verification Pass Rate (%)")
b2 = ax2b.bar(x_idx, sens_mfg_vals, width_b, color="#ef4444", label="E21.0 Hypercalcemia Sensitivity (Mfg RI)")
b3 = ax2b.bar(x_idx + width_b, sens_ref_vals, width_b, color="#16a34a", label="E21.0 Hypercalcemia Sensitivity (refineR RI)")

for bars in [b1, b2, b3]:
    for bar in bars:
        h = bar.get_height()
        ax2b.text(bar.get_x() + bar.get_width() / 2.0, h + 1.5, f"{h:.1f}%", ha="center", va="bottom", fontsize=8.5, fontweight="bold")

ax2b.set_xticks(x_idx)
ax2b.set_xticklabels(["Analyzer Alpha\n(Mfg: 8.6–10.4 | refineR: 8.6–10.4)", "Analyzer Omega\n(Mfg: 8.4–10.5 | refineR: 8.3–10.1)"], fontsize=9, fontweight="bold")
ax2b.set_ylim(0, 114)
ax2b.set_ylabel("Percentage (%)", fontsize=9.5, fontweight="bold")
ax2b.set_title("B. Why 20-Patient Verification Fails: 98.3% Pass Rate Masks Missed E21.0 Cases", fontsize=11, fontweight="bold")
ax2b.legend(loc="upper right", fontsize=8.2, frameon=True)
ax2b.grid(axis="y", linestyle=":", alpha=0.6)

plt.tight_layout()

# Package both figures + all canonical tables into the 4-tab HTML report & open in a new tab
launch_html_dashboard(fig1, fig2, filename="/content/ASCP_Calcium_QA_Dashboard.html")

buf1, buf2 = BytesIO(), BytesIO()
fig1.savefig(buf1, format="png", dpi=145, bbox_inches="tight")
fig2.savefig(buf2, format="png", dpi=145, bbox_inches="tight")
plt.close(fig1)
plt.close(fig2)

display(Image(data=buf2.getvalue()))
display(Image(data=buf1.getvalue()))


## 💾 Optional Take-Home Export

In [ ]:
#@title 📥 Step 5: Download Standalone HTML QA Dashboard & Curated Calcium CSV { display-mode: "form" }
try:
    from google.colab import files
    if os.path.exists("/content/ASCP_Calcium_QA_Dashboard.html"):
        files.download("/content/ASCP_Calcium_QA_Dashboard.html")
    if os.path.exists("/content/ascp_electrolyte_data.csv"):
        files.download("/content/ascp_electrolyte_data.csv")
except Exception:
    print("Files are saved inside your Colab VM at `/content/ASCP_Calcium_QA_Dashboard.html` and `/content/ascp_electrolyte_data.csv`.")
